In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,count
from pyspark.sql.types import StructType, StructField, StringType



CRIME_SCHEMA = StructType(
    [
        StructField("code", StringType()),
        StructField("region", StringType()),
        StructField("category", StringType()),
    ]
)

input_df = spark.readStream.load(
    format="csv",
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/crime_data/input/",
    schema=CRIME_SCHEMA,
)

result_df = input_df.groupBy("region").agg(count("*").alias("total_crime"))

(
    result_df.writeStream.option(
        "checkpointLocation",
        "/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/crime_data_1",
    )
    .trigger(availableNow=True)
    .outputMode("COMPLETE")
    .toTable("merit_catalog.quickstart_schema.crime02")
)